In [ ]:
import sys, os, shutil
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

In [ ]:
import pyvista as pv
import pathlib
import hashlib
import zipfile
import requests
import numpy as np
import matplotlib.pyplot as pl

In [ ]:
import fenics_sz.utils
from fenics_sz.sz_problems.sz_params import allsz_params
from fenics_sz.sz_problems.sz_slab import create_slab
from fenics_sz.fluid_release.perple_x_class import PerpleXGrid
from fenics_sz.fluid_release.slab_dehydration_class import SlabDehydration

In [ ]:
zipfilename = pathlib.Path(os.path.join(basedir, os.path.pardir, os.path.pardir, "data", "vankeken_wilson_peps_2023_TF_lowres_minimal.zip"))
if not zipfilename.is_file():
    zipfileurl = 'https://zenodo.org/records/13234021/files/vankeken_wilson_peps_2023_TF_lowres_minimal.zip'
    r = requests.get(zipfileurl, allow_redirects=True)
    open(zipfilename, 'wb').write(r.content)
assert hashlib.md5(open(zipfilename, 'rb').read()).hexdigest() == 'a8eca6220f9bee091e41a680d502fe0d'

In [ ]:
dmm_thickness = 2.0

# negative number implies below slab, positive implies above it
layer_thicknesses = [
                 None, # sediments - set in the loop
                 -0.3,           # upper volcanics
                 -0.3,           # lower volcanics
                 -1.4,           # dikes
                 -5.0,           # gabbro
                 None  # subslab mantle - set in loop
                ]

csv_path = os.path.join(os.pardir, os.pardir, 'data', 'perple_x_v7.1.9', 'abers_25')
layer_h2os = [
    None, # set this in the loop
    PerpleXGrid(csv_file=os.path.join(csv_path, 'upvolc_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'lovolc_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'dike_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'gabbro_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'DMMdamp_25_h2o.csv'))
]

In [ ]:
dmm_thicknesses = [2.0, 12.0]
h2o_data = [dict() for i in range(len(dmm_thicknesses))]
sres = 1.0
tres = 1.0
for name, szdict in allsz_params.items():
    print('sz:', name)
    slab = create_slab(szdict['xs'], szdict['ys'], sres, szdict['lc_depth'])

    tffilename = os.path.join('vankeken_wilson_peps_2023_TF_lowres_minimal', 'sz_suite_td', szdict['dirname']+'_minres_2.00_cfl_2.00.vtu')
    tffilepath = os.path.join(basedir, os.path.pardir, os.path.pardir, 'data')
    with zipfile.ZipFile(zipfilename, 'r') as z:
        z.extract(tffilename, path=tffilepath)
    tfgrid = pv.get_reader(os.path.join(tffilepath, tffilename)).read()

    layer_thicknesses[0] = -szdict['z15']
    layer_h2os[0] = PerpleXGrid(csv_file=os.path.join(csv_path, szdict['sed_type']+'_h2o.csv'))

    for d, dmm_thickness in enumerate(dmm_thicknesses):
        layer_thicknesses[-1] = -dmm_thickness
        slabmodel = SlabDehydration(sres, tres, layer_thicknesses, layer_h2os, layer_tres=None,
                                    slab=slab, Tgrid=tfgrid, 
                                    Tname='Temperature::PotentialTemperature', 
                                    coast_distance=szdict['coast_distance'], 
                                    sztype=szdict['sztype'], lc_depth=szdict['lc_depth'],
                                    trench_length=szdict['trench_length'], Vs=szdict['Vs'])

        print('  dmm_thickness, conservation_error, water_retention = ', dmm_thickness, sum([errors.sum() for errors in slabmodel.conservation_errors]), slabmodel.water_retention)

        indices = np.argsort(-slabmodel.mesh.dof_xys[:,1])
        h2o_data[d][name] = (slabmodel.total_cumulative_H2O_losses/1000.0, slabmodel.mesh.dof_xys[indices,1], slabmodel.water_retention)

In [ ]:
fig, ax = pl.subplots(figsize=(10,15))
for name, loss_data in h2o_data[0].items():
    ax.plot(loss_data[0], -loss_data[1], label=name)
ax.set_ylim((220, 15))
ax.legend(ncols=3)
ax.set_ylabel('depth of water release (km)')
ax.set_xlabel('cumulative water loss (Tg/Myr)')
fig.show()

In [ ]:
fig, ax = pl.subplots(figsize=(10,15))
for name, loss_data in h2o_data[1].items():
    ax.plot(loss_data[0], -loss_data[1], label=name)
ax.set_ylim((220, 15))
ax.legend(ncols=3)
ax.set_ylabel('depth of water release (km)')
ax.set_xlabel('cumulative water loss (Tg/Myr)')
fig.show()

In [ ]:
# Global Water Retention
[(dmm_thicknesses[d], sum([values[2] for values in h2o_data[d].values()])/1e8) for d in range(len(h2o_data))]